# INFO-6147-(01)-26W Deep Learning with Pytorch

## Homework 2: Softmax Classifier 

Student Name: Yun-Jiung Wang

Student Number: 1256222

Date: Feb 17th, 2026



Import required libraries

In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
from sklearn.metrics import classification_report
from tqdm.auto import tqdm


c:\Users\virwa\.conda\envs\dl_pyTorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Check GPU

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


Transform dataset to Tensor and normalize

In [3]:
torch.manual_seed(42)

if device == torch.device("cuda"):
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) # Normalize RGB channels
])

2. 3. Get CIFAR10 dataset and Split into train, test and validate

In [4]:
#Load the CIFAR-10 dataset
full_train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform) 
full_test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Split the training dataset into training and validation sets, and split the test dataset into two equal parts for testing and validation.
train_dataset, val_dataset, _ = random_split(full_train_dataset, [35000, 10000, 5000])
test_dataset,_ = random_split(full_test_dataset, [5000, 5000])

# Generate data loaders, use shuffle to ensure the data distribution in each batch is random to avoid bias during training.
batch_size = 1024
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,num_workers=0, pin_memory=True)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Files already downloaded and verified
Files already downloaded and verified
Train samples: 35000
Validation samples: 10000
Test samples: 5000


Define Softmax classifier = single Linear layer

In [5]:
class SoftmaxClassifier(nn.Module):
    def __init__(self, input_size=3*32*32, num_classes=10):
        super(SoftmaxClassifier, self).__init__()
        self.fc = nn.Linear(input_size, num_classes)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        out = self.fc(x)
        return out

model = SoftmaxClassifier().to(device)
print(model)

SoftmaxClassifier(
  (fc): Linear(in_features=3072, out_features=10, bias=True)
)


CrossEntropyLoss includes softmax internally

In [6]:
# CrossEntropyLoss combines nn.LogSoftmax() and nn.NLLLoss() in one single class. It is useful when training a classification problem with C classes. The input is expected to contain raw, unnormalized scores for each class (often referred to as logits), and the target is expected to contain the class indices in the range [0, C-1].
criterion = nn.CrossEntropyLoss() 
# weight_decay is the L2 regularization term, which helps to prevent overfitting by adding a penalty to the loss function based on the magnitude of the model's weights. A common choice for weight_decay is 0.001, but you can experiment with different values to see how it affects the model's performance.
weight_decay = 0.001 

optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=weight_decay) 

Train Model

In [7]:
def train_model(model, optimizer, epochs=200):
    model.to(device)
    # Use tqdm to create a progress bar for the training loop
    pbar = tqdm(range(epochs), desc="Training Progress")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad() # Clear gradients before backpropagation
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        
        if (epoch+1) % 20 == 0 or epoch == 0:
            epoch_loss = running_loss / len(train_loader.dataset)
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")          
        
    print("Training complete.")   
    return model 

Evaluate model

In [8]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    accuracy = 100 * correct / total
    
    return accuracy, all_targets, all_preds

3. Train a SoftMax classifier for 200 epochs, and report the accuracy on your training set

In [9]:
train_model(model, optimizer)

train_acc,_,_ = evaluate(val_loader)
print(f"Training Accuracy: {train_acc:.4f}%")

Training Progress:   0%|          | 0/200 [00:05<?, ?it/s, loss=2.1507]

Epoch [1/200], Loss: 2.1698


Training Progress:   0%|          | 0/200 [01:32<?, ?it/s, loss=1.6589]

Epoch [20/200], Loss: 1.7477


Training Progress:   0%|          | 0/200 [04:55<?, ?it/s, loss=1.7076]

...

Epoch [160/200], Loss: 1.6935


Training Progress:   0%|          | 0/200 [30:18<?, ?it/s, loss=1.7587]

Epoch [180/200], Loss: 1.6528


Training Progress:   0%|          | 0/200 [32:51<?, ?it/s, loss=1.5513]


Epoch [200/200], Loss: 1.6076
Training complete.
Training Accuracy: 36.8900%


4. Report the accuracy on your Validation set (remember to set the model to eval mode)

In [10]:
val_acc,_,_ = evaluate(val_loader)
print(f"Validation Accuracy: {val_acc:.4f}%")

Validation Accuracy: 36.8900%


5. Report your accuracy on your Test set (remember to set the model to eval mode)

In [11]:
test_acc, y_true, y_pred = evaluate(test_loader)
print(f"Test Accuracy: {test_acc:.4f}%")

Test Accuracy: 36.6000%


6. Test Data Classification Report

In [12]:
# get the class names from the full dataset
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=full_train_dataset.classes))


--- Classification Report ---
              precision    recall  f1-score   support

    airplane       0.49      0.35      0.41       505
  automobile       0.47      0.40      0.43       530
        bird       0.30      0.33      0.31       479
         cat       0.19      0.16      0.17       507
        deer       0.40      0.19      0.25       497
         dog       0.25      0.39      0.30       478
        frog       0.41      0.39      0.40       497
       horse       0.49      0.39      0.43       522
        ship       0.53      0.45      0.49       499
       truck       0.33      0.61      0.43       486

    accuracy                           0.37      5000
   macro avg       0.38      0.37      0.36      5000
weighted avg       0.39      0.37      0.36      5000



7. Compare multiple weight_decay

using epoch =50

In [13]:
weight_decays = [0.0, 0.0001, 0.001, 0.01]
best_val_acc = 0
best_wd = 0
results = {}

for wd in weight_decays:
    
    # Reinitialize the model and optimizer for each weight decay value
    model_exp = SoftmaxClassifier().to(device)
    
    # Reinitialize the optimizer with the new model parameters and weight decay
    optimizer_exp = optim.SGD(model_exp.parameters(), lr=0.04, weight_decay=wd)
    
    # Implement the training loop for the current weight decay value
    train_model(model_exp, optimizer_exp, epochs=50) # Use epchos=50 for quicker experimentation
    
    # Evaluate the model on the validation set and store the accuracy  
    val_acc,_,_ = evaluate(val_loader)
    results[wd] = val_acc
    
    print(f"Training with weight decay: {wd}, Validation Accuracy: {val_acc:.4f}")
    
best_wd = max(results, key=results.get)    
print(f"Best weight decay: {best_wd}, Validation Accuracy: {results[best_wd]:.4f}")

Training Progress:   0%|          | 0/50 [00:04<?, ?it/s, loss=1.8522]

Epoch [1/50], Loss: 1.9600


Training Progress:   0%|          | 0/50 [01:28<?, ?it/s, loss=1.6759]

Epoch [20/50], Loss: 1.6842


Training Progress:   0%|          | 0/50 [02:57<?, ?it/s, loss=1.6851]

...

Epoch [20/50], Loss: 1.6943


Training Progress:   0%|          | 0/50 [08:38<?, ?it/s, loss=1.6611]

Epoch [40/50], Loss: 1.6653


Training Progress:   0%|          | 0/50 [10:37<?, ?it/s, loss=1.6938]


Training complete.
Training with weight decay: 0.01, Validation Accuracy: 36.8900
Best weight decay: 0.0, Validation Accuracy: 36.8900


Training final model with best weight decay

In [14]:
final_model = SoftmaxClassifier().to(device)
final_optimizer = optim.SGD(
    final_model.parameters(),
    lr=0.04,
    weight_decay=best_wd
)

final_model=train_model(final_model, final_optimizer,epochs=200)

test_acc, y_true, y_pred = evaluate(test_loader)
print(f"Final Test Accuracy: {test_acc:.4f}")

Training Progress:   0%|          | 0/200 [00:05<?, ?it/s, loss=1.9065]

Epoch [1/200], Loss: 1.9612


Training Progress:   0%|          | 0/200 [01:34<?, ?it/s, loss=1.6674]

Epoch [20/200], Loss: 1.6849


Training Progress:   0%|          | 0/200 [03:03<?, ?it/s, loss=1.6532]

...

Epoch [160/200], Loss: 1.5879


Training Progress:   0%|          | 0/200 [21:02<?, ?it/s, loss=1.5787]

Epoch [180/200], Loss: 1.5825


Training Progress:   0%|          | 0/200 [24:19<?, ?it/s, loss=1.5660]


Epoch [200/200], Loss: 1.5771
Training complete.
Final Test Accuracy: 36.6000


Evaluate all sets

In [16]:
model= final_model

train_acc, y_true_train, y_pred_train = evaluate(train_loader)
val_acc, y_true_val, y_pred_val = evaluate(val_loader)
test_acc, y_true_test, y_pred_test = evaluate(test_loader)

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 46.465714285714284
Validation Accuracy: 40.04
Test Accuracy: 39.98


Classification Report

In [19]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        # make sure to move predictions and labels back to CPU before converting to numpy
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
# Add label names to the classification report for better readability. 
report = classification_report(
    all_labels, 
    all_preds, 
    target_names=full_train_dataset.classes,
    digits=4 
)

print(report)

              precision    recall  f1-score   support

    airplane     0.4610    0.4218    0.4405       505
  automobile     0.5072    0.4000    0.4473       530
        bird     0.3194    0.2401    0.2741       479
         cat     0.2699    0.2939    0.2814       507
        deer     0.3641    0.4366    0.3971       497
         dog     0.2872    0.2950    0.2910       478
        frog     0.4674    0.4044    0.4337       497
       horse     0.4707    0.4157    0.4415       522
        ship     0.4911    0.5531    0.5203       499
       truck     0.3862    0.5309    0.4471       486

    accuracy                         0.3998      5000
   macro avg     0.4024    0.3991    0.3974      5000
weighted avg     0.4041    0.3998    0.3986      5000

